# A0 — Prescribed

**Static, single-agent DAG.** No dynamic decisions. The runtime just runs the declared agent and returns its output.

**Fictional task**: a three-line haiku about a lost robot finding a garden.

In [ ]:
# --- Load API key from the canonical env file (see memory `reference_api_keys`) ---
import os
from pathlib import Path

env_file = Path('/home/shumway/projects/meta-agents/.env')
if env_file.exists() and not os.environ.get('OPENROUTER_API_KEY'):
    for raw in env_file.read_text().splitlines():
        s = raw.strip()
        if s.startswith('OPENROUTER_API_KEY='):
            os.environ['OPENROUTER_API_KEY'] = s.split('=', 1)[1].strip().strip('"').strip("'")
            break

assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY missing'
# Default worker model for agents that do not declare their own (DAG engine consults LLM_MODEL).
os.environ.setdefault('LLM_MODEL', 'deepseek/deepseek-chat-v3.1')
print('OpenRouter key loaded. Default model:', os.environ['LLM_MODEL'])

## Load the A0 workflow

We reuse the `01-hello-world` example — a one-agent DAG — and give it a fresh fictional prompt.

In [ ]:
from pathlib import Path
from awp.parser import parse_manifest
from awp.validator import check_compliance, AutonomyLevel
from awp.parser import parse_agent

WORKFLOW_DIR = Path('/home/shumway/projects/agent-workflow-protocol/examples/workflows/01-hello-world')
manifest = parse_manifest(WORKFLOW_DIR / 'workflow.awp.yaml')
agents = {}
for ad in (WORKFLOW_DIR / 'agents').iterdir():
    a = ad / 'agent.awp.yaml'
    if a.exists():
        agents[ad.name] = parse_agent(a)

result = check_compliance(manifest, agents, target_level=AutonomyLevel.A0_PRESCRIBED)
assert result.level >= AutonomyLevel.A0_PRESCRIBED, f'Not A0: {result.errors}'
print(f'Compliance: {result.level_name} ({result.level.name})')

## Run the workflow

A0 uses the DAG engine — a single topological pass. The agent returns a dict with a `confidence` field (R17).

In [ ]:
import json
import logging
logging.basicConfig(level=logging.WARNING)

from awp.runtime import WorkflowRunner

TASK = 'Write a three-line haiku about a lost robot that finds a hidden garden.'
runner = WorkflowRunner(
    WORKFLOW_DIR,
    worker_model='deepseek/deepseek-chat-v3.1',
)
result = runner.run(TASK)

print(json.dumps(result, indent=2, default=str)[:2000])

## Assertions (E2E rubric)

In [ ]:
assert isinstance(result, dict), 'result must be a dict'
# Filter to dict-typed agent outputs (the runner also echoes the task string)
agent_outputs = {
    k: v for k, v in result.items()
    if not k.startswith('_') and isinstance(v, dict) and 'confidence' in v
}
assert agent_outputs, f'no agent outputs found. Keys: {list(result)}'
for aid, output in agent_outputs.items():
    conf = output['confidence']
    assert isinstance(conf, (int, float)) and 0.0 <= conf <= 1.0, f'{aid} confidence invalid'
    assert not output.get('error'), f'{aid} failed: {output["error"]}'
print(f'OK — agent outputs: {list(agent_outputs)}')